# MambaBTS — Brain Tumor Segmentation on BraTS 2021
**Dataset:** `dschettler8845/brats-2021-task1`

Attach the dataset in Kaggle settings before running:
- `BraTS2021_Training_Data.tar` (full training set)
- `BraTS2021_00495.tar` / `BraTS2021_00621.tar` (held-out test subjects)

Enable GPU accelerator: *Settings → Accelerator → GPU T4 x2* (or P100).

## 0. Install dependencies

In [ ]:
%%capture
!pip install nibabel scipy scikit-learn --quiet
# antspyx is optional (N4 bias correction) — large wheel, skip if slow
# !pip install antspyx --quiet

## 1. Extract BraTS 2021 archives

In [ ]:
import os, tarfile, glob

INPUT_DIR  = "/kaggle/input/datasets/dschettler8845/brats-2021-task1"
WORK_DIR   = "/kaggle/working"
DATA_DIR   = os.path.join(WORK_DIR, "brats2021")
TEST_DIR   = os.path.join(WORK_DIR, "brats2021_test")
CKPT_DIR   = os.path.join(WORK_DIR, "checkpoints")

os.makedirs(DATA_DIR,  exist_ok=True)
os.makedirs(TEST_DIR,  exist_ok=True)
os.makedirs(CKPT_DIR,  exist_ok=True)

# ── Extract main training archive ────────────────────────────────────────────
train_tar = os.path.join(INPUT_DIR, "BraTS2021_Training_Data.tar")
if os.path.exists(train_tar):
    print(f"Extracting {train_tar} ...")
    with tarfile.open(train_tar, "r") as tf:
        tf.extractall(DATA_DIR)
    print("Done.")
else:
    print(f"WARNING: {train_tar} not found — check dataset attachment.")

# ── Extract held-out test subjects ───────────────────────────────────────────
for fname in ["BraTS2021_00495.tar", "BraTS2021_00621.tar"]:
    fpath = os.path.join(INPUT_DIR, fname)
    if os.path.exists(fpath):
        print(f"Extracting {fname} ...")
        with tarfile.open(fpath, "r") as tf:
            tf.extractall(TEST_DIR)
        print("Done.")
    else:
        print(f"WARNING: {fpath} not found.")

# ── Verify dataset structure ──────────────────────────────────────────────────
subjects = sorted([
    d for d in os.listdir(DATA_DIR)
    if os.path.isdir(os.path.join(DATA_DIR, d))
])
print(f"\nFound {len(subjects)} training subjects.")
if subjects:
    print("First subject files:")
    first = os.path.join(DATA_DIR, subjects[0])
    for f in sorted(os.listdir(first)):
        print(f"  {f}")

## 2. Normalise directory layout

BraTS 2021 files use the naming convention:
`BraTS2021_XXXXX_t1.nii.gz`, `_t1ce.nii.gz`, `_t2.nii.gz`, `_flair.nii.gz`, `_seg.nii.gz`

The `BraTSDataset` loader in `mamba_bts_impl.py` expects exactly this layout — no action required.
The cell below validates a sample subject to confirm.

In [ ]:
import nibabel as nib

MODALITIES = ["t1", "t1ce", "t2", "flair"]

def check_subject(subj_dir: str):
    name = os.path.basename(subj_dir)
    ok = True
    for mod in MODALITIES + ["seg"]:
        p = os.path.join(subj_dir, f"{name}_{mod}.nii.gz")
        if not os.path.exists(p):
            print(f"  MISSING: {p}")
            ok = False
    if ok:
        img = nib.load(os.path.join(subj_dir, f"{name}_t1.nii.gz"))
        print(f"  {name}: shape={img.shape}, voxel size={img.header.get_zooms()}")

if subjects:
    print("Checking first 3 subjects:")
    for s in subjects[:3]:
        check_subject(os.path.join(DATA_DIR, s))

test_subjects = sorted([
    d for d in os.listdir(TEST_DIR)
    if os.path.isdir(os.path.join(TEST_DIR, d))
])
print(f"\nTest subjects ({len(test_subjects)}):")
for s in test_subjects:
    check_subject(os.path.join(TEST_DIR, s))

## 3. Write model source & run smoke test

The full `mamba_bts_impl.py` source is embedded directly in the cell below using `%%writefile`, so **no file upload is needed** — the notebook is completely self-contained.


In [ ]:
%%writefile /kaggle/working/mamba_bts_impl.py
"""
MambaBTS: State Space Model-based Brain Tumor Segmentation
Based on: "Deep learning for brain tumor segmentation in multimodal MRI images" (2025)

Dataset: BraTS format — 4 MRI modalities: T1, T1ce, T2, FLAIR
Labels:  0=Background, 1=Necrotic/Non-Enhancing Tumor Core (NCR/NET),
         2=Peritumoral Edema (ED), 3=Enhancing Tumor (ET)
"""

from __future__ import annotations

import math
import os
import warnings
from typing import List, Optional, Tuple

import nibabel as nib
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.ndimage import binary_fill_holes, label
from scipy.spatial.distance import directed_hausdorff
from torch.utils.checkpoint import checkpoint as grad_ckpt
from torch.utils.data import DataLoader, Dataset

warnings.filterwarnings("ignore", category=UserWarning)

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 1 — IMAGE PROCESSING PIPELINE
# ─────────────────────────────────────────────────────────────────────────────


class N4BiasFieldCorrection:

    def __init__(self, n_iterations: List[int] = [50, 50, 50, 50],
                 convergence_threshold: float = 1e-6):
        self.n_iterations = n_iterations
        self.convergence_threshold = convergence_threshold
        self._use_ants = self._check_ants()

    @staticmethod
    def _check_ants() -> bool:
        try:
            import ants  # noqa: F401
            return True
        except ImportError:
            return False

    def correct(self, image: np.ndarray,
                mask: Optional[np.ndarray] = None) -> np.ndarray:
        if self._use_ants:
            return self._correct_ants(image, mask)
        return self._correct_approx(image, mask)

    def _correct_ants(self, image: np.ndarray,
                      mask: Optional[np.ndarray]) -> np.ndarray:
        import ants
        ants_img = ants.from_numpy(image.astype(np.float32))
        ants_mask = (
            ants.from_numpy(mask.astype(np.float32))
            if mask is not None
            else ants.get_mask(ants_img)
        )
        corrected = ants.n4_bias_field_correction(
            ants_img,
            mask=ants_mask,
            convergence={"iters": self.n_iterations,
                         "tol": self.convergence_threshold},
        )
        return corrected.numpy()

    @staticmethod
    def _correct_approx(image: np.ndarray,
                        mask: Optional[np.ndarray]) -> np.ndarray:
        from scipy.ndimage import gaussian_filter

        img = image.astype(np.float64)
        img = np.where(img > 0, img, np.finfo(np.float64).eps)
        log_img = np.log(img)
        bias_field = gaussian_filter(log_img, sigma=10)
        corrected = np.exp(log_img - bias_field)
        scale = np.median(img[img > 0]) / (np.median(corrected[corrected > 0]) + 1e-8)
        corrected = corrected * scale
        return corrected.astype(image.dtype)


class ZScoreNormalization:

    def __init__(self, mask_threshold: float = 0.0):
        self.mask_threshold = mask_threshold

    def normalize(self, image: np.ndarray,
                  mask: Optional[np.ndarray] = None) -> np.ndarray:
        if mask is None:
            mask = image > self.mask_threshold
        brain_voxels = image[mask]
        mean = brain_voxels.mean()
        std = brain_voxels.std() + 1e-8
        normalized = np.zeros_like(image, dtype=np.float32)
        normalized[mask] = (image[mask] - mean) / std
        return normalized


class SkullStripper:

    def __init__(self, method: str = "auto"):
        self.method = method

    def strip(self, image: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        if self.method in ("auto", "ants") and self._check_ants():
            return self._strip_ants(image)
        if self.method in ("auto", "threshold"):
            return self._strip_threshold(image)
        raise ValueError(f"Unknown skull-stripping method: {self.method}")

    @staticmethod
    def _check_ants() -> bool:
        try:
            import antspynet  # noqa: F401
            return True
        except ImportError:
            return False

    @staticmethod
    def _strip_ants(image: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        import ants
        import antspynet
        ants_img = ants.from_numpy(image.astype(np.float32))
        prob = antspynet.brain_extraction(ants_img, modality="t1")
        mask = (prob.numpy() > 0.5).astype(np.uint8)
        stripped = image * mask
        return stripped.astype(np.float32), mask

    @staticmethod
    def _strip_threshold(image: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        from scipy.ndimage import binary_closing, binary_erosion, generate_binary_structure

        threshold = image.max() * 0.1
        binary = (image > threshold).astype(bool)
        struct = generate_binary_structure(3, 2)
        closed = binary_closing(binary, structure=struct, iterations=3)
        filled = binary_fill_holes(closed)
        labeled, n_components = label(filled)
        if n_components == 0:
            return image.astype(np.float32), filled.astype(np.uint8)
        sizes = [np.sum(labeled == i) for i in range(1, n_components + 1)]
        largest = np.argmax(sizes) + 1
        brain_mask = (labeled == largest).astype(np.uint8)
        brain_mask = binary_erosion(brain_mask, structure=struct,
                                    iterations=2).astype(np.uint8)
        stripped = image * brain_mask
        return stripped.astype(np.float32), brain_mask


class BraTSPreprocessor:

    def __init__(self,
                 n4_iterations: List[int] = [50, 50, 50, 50],
                 skull_strip_method: str = "auto"):
        self.n4 = N4BiasFieldCorrection(n_iterations=n4_iterations)
        self.skull_stripper = SkullStripper(method=skull_strip_method)
        self.normalizer = ZScoreNormalization()

    def preprocess_subject(
        self,
        modality_paths: dict,
        seg_path: Optional[str] = None,
    ) -> Tuple[np.ndarray, Optional[np.ndarray]]:
        modality_order = ["t1", "t1ce", "t2", "flair"]
        volumes = {}
        for mod in modality_order:
            img_nii = nib.load(modality_paths[mod])
            data = img_nii.get_fdata(dtype=np.float32)
            corrected = self.n4.correct(data)
            volumes[mod] = corrected
        _, brain_mask = self.skull_stripper.strip(volumes["t1ce"])
        processed = []
        for mod in modality_order:
            norm = self.normalizer.normalize(volumes[mod],
                                             mask=brain_mask.astype(bool))
            processed.append(norm)
        volume = np.stack(processed, axis=0)
        seg = None
        if seg_path is not None:
            seg_nii = nib.load(seg_path)
            seg = seg_nii.get_fdata(dtype=np.float32).astype(np.uint8)
        return volume, seg


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 2 — MAMBA BLOCKS & MAMBABTS ARCHITECTURE
# ─────────────────────────────────────────────────────────────────────────────


class SelectiveSSM(nn.Module):

    def __init__(self, d_model: int, d_state: int = 16, dt_rank: int = None):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.dt_rank = dt_rank or math.ceil(d_model / 16)

        self.x_proj = nn.Linear(d_model, self.dt_rank + 2 * d_state, bias=False)
        self.dt_proj = nn.Linear(self.dt_rank, d_model, bias=True)

        A = torch.arange(1, d_state + 1, dtype=torch.float32).repeat(d_model, 1)
        self.A_log = nn.Parameter(torch.log(A))
        self.D = nn.Parameter(torch.ones(d_model))

        dt_init_std = self.dt_rank ** -0.5
        nn.init.uniform_(self.dt_proj.weight, -dt_init_std, dt_init_std)
        dt = torch.exp(torch.rand(d_model) * (math.log(0.1) - math.log(0.001))
                       + math.log(0.001)).clamp(min=1e-4)
        inv_dt = dt + torch.log(-torch.expm1(-dt))
        with torch.no_grad():
            self.dt_proj.bias.copy_(inv_dt)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, L, D = x.shape
        N = self.d_state

        x_proj = self.x_proj(x)
        dt_raw, B_mat, C_mat = x_proj.split([self.dt_rank, N, N], dim=-1)
        dt = F.softplus(self.dt_proj(dt_raw))

        A = -torch.exp(self.A_log.float())  # (D, N)
        # Compute dA/dB per-step to avoid materialising (B, L, D, N) tensors.
        h = torch.zeros(B, D, N, device=x.device, dtype=x.dtype)
        ys = []
        for t in range(L):
            dt_t = dt[:, t].unsqueeze(-1)            # (B, D, 1)
            dA_t = torch.exp(dt_t * A)               # (B, D, N)
            dB_t = dt_t * B_mat[:, t].unsqueeze(1)   # (B, D, N)
            h = dA_t * h + dB_t * x[:, t].unsqueeze(-1)
            y_t = (h * C_mat[:, t].unsqueeze(1)).sum(dim=-1)
            ys.append(y_t)

        y = torch.stack(ys, dim=1)
        y = y + x * self.D
        return y


class MambaBlock(nn.Module):

    def __init__(self, d_model: int, d_state: int = 16,
                 d_conv: int = 4, expand: int = 2,
                 dt_rank: Optional[int] = None):
        super().__init__()
        self.d_inner = int(expand * d_model)
        self.norm = nn.LayerNorm(d_model)
        self.in_proj = nn.Linear(d_model, 2 * self.d_inner, bias=False)
        self.conv1d = nn.Conv1d(
            in_channels=self.d_inner,
            out_channels=self.d_inner,
            kernel_size=d_conv,
            padding=d_conv - 1,
            groups=self.d_inner,
            bias=True,
        )
        self.ssm = SelectiveSSM(self.d_inner, d_state=d_state, dt_rank=dt_rank)
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = x
        x = self.norm(x)
        xz = self.in_proj(x)
        x_main, z = xz.chunk(2, dim=-1)
        x_conv = self.conv1d(x_main.transpose(1, 2))
        x_conv = x_conv[..., :x_main.shape[1]]
        x_conv = x_conv.transpose(1, 2)
        x_conv = F.silu(x_conv)
        y = self.ssm(x_conv)
        y = y * F.silu(z)
        out = self.out_proj(y)
        return out + residual


class MambaBlock3D(nn.Module):

    def __init__(self, channels: int, d_state: int = 16,
                 d_conv: int = 4, expand: int = 2):
        super().__init__()
        self.channels = channels
        self.mamba_z = MambaBlock(channels, d_state=d_state, d_conv=d_conv, expand=expand)
        self.mamba_y = MambaBlock(channels, d_state=d_state, d_conv=d_conv, expand=expand)
        self.mamba_x = MambaBlock(channels, d_state=d_state, d_conv=d_conv, expand=expand)
        self.fusion = nn.Conv3d(channels, channels, kernel_size=1, bias=False)
        self.norm = nn.GroupNorm(8, channels)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C, H, W, D = x.shape
        # Gradient checkpointing on each SSM scan: recomputes activations on
        # backward instead of storing them, so both T4s carry equal load.
        xz = x.permute(0, 2, 3, 1, 4).reshape(B * H * W, D, C)
        yz = grad_ckpt(self.mamba_z, xz, use_reentrant=False).reshape(B, H, W, D, C).permute(0, 4, 1, 2, 3)
        xy = x.permute(0, 3, 4, 1, 2).reshape(B * W * D, H, C)
        yy = grad_ckpt(self.mamba_y, xy, use_reentrant=False).reshape(B, W, D, H, C).permute(0, 4, 3, 1, 2)
        xx = x.permute(0, 2, 4, 1, 3).reshape(B * H * D, W, C)
        yx = grad_ckpt(self.mamba_x, xx, use_reentrant=False).reshape(B, H, D, W, C).permute(0, 4, 1, 3, 2)
        fused = self.fusion(yz + yy + yx)
        return F.relu(self.norm(fused))


class ConvBlock3D(nn.Module):

    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.GroupNorm(min(8, out_ch), out_ch),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.GroupNorm(min(8, out_ch), out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class EncoderBlock(nn.Module):

    def __init__(self, in_ch: int, out_ch: int, d_state: int = 16):
        super().__init__()
        self.conv = ConvBlock3D(in_ch, out_ch)
        self.mamba = MambaBlock3D(out_ch, d_state=d_state)
        self.pool = nn.MaxPool3d(kernel_size=2, stride=2)

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        skip = self.mamba(self.conv(x))
        return self.pool(skip), skip


class DecoderBlock(nn.Module):

    def __init__(self, in_ch: int, out_ch: int, d_state: int = 16):
        super().__init__()
        self.up = nn.ConvTranspose3d(in_ch, out_ch, kernel_size=2, stride=2)
        self.conv = ConvBlock3D(out_ch * 2, out_ch)
        self.mamba = MambaBlock3D(out_ch, d_state=d_state)

    def forward(self, x: torch.Tensor, skip: torch.Tensor) -> torch.Tensor:
        x = self.up(x)
        if x.shape != skip.shape:
            x = F.interpolate(x, size=skip.shape[2:], mode="trilinear",
                              align_corners=False)
        x = torch.cat([x, skip], dim=1)
        return self.mamba(self.conv(x))


class MambaBTS(nn.Module):

    def __init__(self, in_channels: int = 4, num_classes: int = 4,
                 base_features: int = 32, d_state: int = 16):
        super().__init__()
        f = base_features
        self.enc1 = EncoderBlock(in_channels, f,     d_state=d_state)
        self.enc2 = EncoderBlock(f,           f * 2, d_state=d_state)
        self.enc3 = EncoderBlock(f * 2,       f * 4, d_state=d_state)
        self.enc4 = EncoderBlock(f * 4,       f * 8, d_state=d_state)
        self.bottleneck = nn.Sequential(
            ConvBlock3D(f * 8, f * 16),
            MambaBlock3D(f * 16, d_state=d_state),
        )
        self.dec4 = DecoderBlock(f * 16, f * 8,  d_state=d_state)
        self.dec3 = DecoderBlock(f * 8,  f * 4,  d_state=d_state)
        self.dec2 = DecoderBlock(f * 4,  f * 2,  d_state=d_state)
        self.dec1 = DecoderBlock(f * 2,  f,      d_state=d_state)
        self.seg_head = nn.Conv3d(f, num_classes, kernel_size=1)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv3d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out",
                                        nonlinearity="relu")
            elif isinstance(m, (nn.GroupNorm, nn.LayerNorm)):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x, s1 = self.enc1(x)
        x, s2 = self.enc2(x)
        x, s3 = self.enc3(x)
        x, s4 = self.enc4(x)
        x = self.bottleneck(x)
        x = self.dec4(x, s4)
        x = self.dec3(x, s3)
        x = self.dec2(x, s2)
        x = self.dec1(x, s1)
        return self.seg_head(x)


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 3 — DATASET
# ─────────────────────────────────────────────────────────────────────────────


class BraTSDataset(Dataset):

    MODALITIES = ["t1", "t1ce", "t2", "flair"]

    def __init__(self, root: str,
                 patch_size: Tuple[int, int, int] = (96, 96, 96),
                 preprocess: bool = True,
                 augment: bool = False):
        self.root = root
        self.patch_size = patch_size
        self.augment = augment
        self.preprocessor = BraTSPreprocessor() if preprocess else None
        self.subjects = self._build_subject_list()

    def _build_subject_list(self) -> List[dict]:
        subjects = []
        for subj_dir in sorted(os.listdir(self.root)):
            full_path = os.path.join(self.root, subj_dir)
            if not os.path.isdir(full_path):
                continue
            entry = {"name": subj_dir}
            for mod in self.MODALITIES:
                fname = os.path.join(full_path, f"{subj_dir}_{mod}.nii.gz")
                if not os.path.exists(fname):
                    break
                entry[mod] = fname
            else:
                seg_path = os.path.join(full_path, f"{subj_dir}_seg.nii.gz")
                entry["seg"] = seg_path if os.path.exists(seg_path) else None
                subjects.append(entry)
        return subjects

    def __len__(self) -> int:
        return len(self.subjects)

    def __getitem__(self, idx: int) -> dict:
        subj = self.subjects[idx]
        modality_paths = {m: subj[m] for m in self.MODALITIES}
        if self.preprocessor is not None:
            volume, seg = self.preprocessor.preprocess_subject(
                modality_paths, seg_path=subj.get("seg")
            )
        else:
            vols = [nib.load(modality_paths[m]).get_fdata(dtype=np.float32)
                    for m in self.MODALITIES]
            volume = np.stack(vols, axis=0)
            seg = (nib.load(subj["seg"]).get_fdata(dtype=np.float32).astype(np.uint8)
                   if subj.get("seg") else None)
        volume = self._resize_volume(volume)
        if seg is not None:
            seg = self._resize_volume(seg[None])[0]
        volume_tensor = torch.from_numpy(volume)
        result = {"image": volume_tensor, "name": subj["name"]}
        if seg is not None:
            result["seg"] = torch.from_numpy(seg).long()
        return result

    def _resize_volume(self, arr: np.ndarray) -> np.ndarray:
        target = self.patch_size
        spatial = arr.shape[-3:]
        slices_in, slices_out = [], []
        padded = np.zeros(arr.shape[:-3] + target, dtype=arr.dtype)
        for s, t in zip(spatial, target):
            if s >= t:
                start = (s - t) // 2
                slices_in.append(slice(start, start + t))
                slices_out.append(slice(0, t))
            else:
                pad = (t - s) // 2
                slices_in.append(slice(0, s))
                slices_out.append(slice(pad, pad + s))
        padded[..., slices_out[0], slices_out[1], slices_out[2]] = \
            arr[..., slices_in[0], slices_in[1], slices_in[2]]
        return padded


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 4 — LOSS FUNCTIONS
# ─────────────────────────────────────────────────────────────────────────────


class DiceLoss(nn.Module):

    def __init__(self, num_classes: int = 4, smooth: float = 1e-5,
                 ignore_background: bool = True):
        super().__init__()
        self.num_classes = num_classes
        self.smooth = smooth
        self.start_class = 1 if ignore_background else 0

    def forward(self, logits: torch.Tensor,
                targets: torch.Tensor) -> torch.Tensor:
        probs = F.softmax(logits, dim=1)
        targets_onehot = F.one_hot(targets, self.num_classes)
        targets_onehot = targets_onehot.permute(0, 4, 1, 2, 3).float()
        dice_scores = []
        for c in range(self.start_class, self.num_classes):
            p = probs[:, c]
            g = targets_onehot[:, c]
            intersection = (p * g).sum()
            dice = (2 * intersection + self.smooth) / \
                   (p.sum() + g.sum() + self.smooth)
            dice_scores.append(dice)
        return 1.0 - torch.stack(dice_scores).mean()


class CombinedLoss(nn.Module):

    def __init__(self, num_classes: int = 4, ce_weight: float = 0.5):
        super().__init__()
        self.dice = DiceLoss(num_classes)
        self.ce = nn.CrossEntropyLoss()
        self.ce_weight = ce_weight

    def forward(self, logits: torch.Tensor,
                targets: torch.Tensor) -> torch.Tensor:
        return self.dice(logits, targets) + self.ce_weight * self.ce(logits, targets)


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 5 — EVALUATION METRICS
# ─────────────────────────────────────────────────────────────────────────────


def dice_similarity_coefficient(
    pred: np.ndarray,
    target: np.ndarray,
    num_classes: int = 4,
    ignore_background: bool = True,
) -> dict:
    results = {}
    smooth = 1e-8
    start = 1 if ignore_background else 0
    for c in range(start, num_classes):
        p = (pred == c).astype(float)
        g = (target == c).astype(float)
        intersection = (p * g).sum()
        dsc = (2 * intersection + smooth) / (p.sum() + g.sum() + smooth)
        results[f"class_{c}"] = float(dsc)

    def _dsc(p_mask, g_mask):
        i = (p_mask & g_mask).sum()
        return float((2 * i + smooth) / (p_mask.sum() + g_mask.sum() + smooth))

    results["WT"] = _dsc(pred >= 1, target >= 1)
    results["TC"] = _dsc(np.isin(pred, [1, 3]), np.isin(target, [1, 3]))
    results["ET"] = _dsc(pred == 3, target == 3)
    results["mean_dsc"] = float(np.mean([results["WT"], results["TC"], results["ET"]]))
    return results


def hausdorff_distance_95(
    pred: np.ndarray,
    target: np.ndarray,
    num_classes: int = 4,
    percentile: float = 95.0,
    voxel_spacing: Tuple[float, float, float] = (1.0, 1.0, 1.0),
) -> dict:
    from scipy.ndimage import distance_transform_edt

    results = {}
    spacing = np.array(voxel_spacing)

    def _hd95(p_mask: np.ndarray, g_mask: np.ndarray) -> float:
        if p_mask.sum() == 0 and g_mask.sum() == 0:
            return 0.0
        if p_mask.sum() == 0 or g_mask.sum() == 0:
            return float("inf")
        p_surface = p_mask ^ binary_fill_holes(p_mask)
        g_surface = g_mask ^ binary_fill_holes(g_mask)
        dt_p = distance_transform_edt(~p_surface, sampling=spacing)
        dt_g = distance_transform_edt(~g_surface, sampling=spacing)
        d_p2g = dt_g[p_surface]
        d_g2p = dt_p[g_surface]
        all_dist = np.concatenate([d_p2g, d_g2p])
        return float(np.percentile(all_dist, percentile))

    for c in range(1, num_classes):
        results[f"class_{c}"] = _hd95(pred == c, target == c)
    results["WT"] = _hd95(pred >= 1, target >= 1)
    results["TC"] = _hd95(np.isin(pred, [1, 3]), np.isin(target, [1, 3]))
    results["ET"] = _hd95(pred == 3, target == 3)
    return results


def evaluate_batch(
    model: nn.Module,
    dataloader: DataLoader,
    device: torch.device,
    num_classes: int = 4,
) -> dict:
    model.eval()
    all_dsc, all_hd95 = [], []
    with torch.no_grad():
        for batch in dataloader:
            images = batch["image"].to(device)
            segs = batch.get("seg")
            logits = model(images)
            preds = logits.argmax(dim=1).cpu().numpy()
            if segs is not None:
                segs_np = segs.cpu().numpy()
                for b in range(preds.shape[0]):
                    dsc = dice_similarity_coefficient(preds[b], segs_np[b], num_classes)
                    hd = hausdorff_distance_95(preds[b], segs_np[b], num_classes)
                    all_dsc.append(dsc)
                    all_hd95.append(hd)
    if not all_dsc:
        return {}
    mean_dsc = {k: float(np.mean([d[k] for d in all_dsc if k in d]))
                for k in all_dsc[0]}
    mean_hd95 = {k: float(np.mean([h[k] for h in all_hd95
                                   if k in h and not np.isinf(h[k])]))
                 for k in all_hd95[0]}
    return {"DSC": mean_dsc, "HD95": mean_hd95}


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 6 — TRAINING LOOP
# ─────────────────────────────────────────────────────────────────────────────


def train_one_epoch(
    model: nn.Module,
    dataloader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    device: torch.device,
    scaler: Optional[torch.cuda.amp.GradScaler] = None,
) -> float:
    model.train()
    total_loss = 0.0
    for batch in dataloader:
        images = batch["image"].to(device)
        segs = batch["seg"].to(device)
        optimizer.zero_grad()
        if scaler is not None:
            with torch.cuda.amp.autocast():
                logits = model(images)
                loss = criterion(logits, segs)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(images)
            loss = criterion(logits, segs)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
        total_loss += loss.item()
    return total_loss / max(len(dataloader), 1)


def train(
    data_root: str,
    num_epochs: int = 100,
    batch_size: int = 1,
    lr: float = 1e-4,
    patch_size: Tuple[int, int, int] = (96, 96, 96),
    num_classes: int = 4,
    base_features: int = 32,
    d_state: int = 16,
    checkpoint_dir: str = "checkpoints",
    device_str: str = "auto",
):
    os.makedirs(checkpoint_dir, exist_ok=True)

    # ── Device selection ──────────────────────────────────────────────────────
    if device_str == "auto":
        if torch.cuda.is_available():
            device = torch.device("cuda")
        elif torch.backends.mps.is_available():
            device = torch.device("mps")
        else:
            device = torch.device("cpu")
    else:
        device = torch.device(device_str)
    print(f"[Train] Using device: {device}")

    # ── Dataset & DataLoader ──────────────────────────────────────────────────
    dataset = BraTSDataset(data_root, patch_size=patch_size,
                           preprocess=True, augment=True)
    n_val = max(1, int(0.1 * len(dataset)))
    n_train = len(dataset) - n_val
    train_set, val_set = torch.utils.data.random_split(dataset, [n_train, n_val])

    train_loader = DataLoader(train_set, batch_size=batch_size,
                              shuffle=True, num_workers=4, pin_memory=True)
    val_loader = DataLoader(val_set, batch_size=batch_size,
                            shuffle=False, num_workers=4, pin_memory=True)

    # ── Model ─────────────────────────────────────────────────────────────────
    model = MambaBTS(in_channels=4, num_classes=num_classes,
                     base_features=base_features, d_state=d_state).to(device)

    # ── Multi-GPU: DataParallel with explicit device_ids ──────────────────────
    # device_ids must be set explicitly so PyTorch won't silently fall back
    # to GPU 0 only when CUDA_VISIBLE_DEVICES is unset in the Kaggle env.
    n_gpus = torch.cuda.device_count() if device.type == "cuda" else 0
    if n_gpus > 1:
        device_ids = list(range(n_gpus))
        print(f"[Train] Using DataParallel across GPUs {device_ids}")
        model = nn.DataParallel(model, device_ids=device_ids)
        for i in device_ids:
            free, total = torch.cuda.mem_get_info(i)
            print(f"  GPU {i}: {free // 1024**2} MB free / {total // 1024**2} MB total")
    else:
        print(f"[Train] Single GPU / CPU — no DataParallel")

    # ── Loss, optimiser, scheduler, AMP scaler ────────────────────────────────
    criterion = CombinedLoss(num_classes=num_classes)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=num_epochs, eta_min=1e-6
    )
    scaler = torch.cuda.amp.GradScaler() if device.type == "cuda" else None

    best_dsc = 0.0
    for epoch in range(1, num_epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer,
                                     criterion, device, scaler)
        scheduler.step()

        if epoch % 10 == 0 or epoch == num_epochs:
            metrics = evaluate_batch(model, val_loader, device, num_classes)
            mean_dsc = metrics.get("DSC", {}).get("mean_dsc", 0.0)
            print(f"[Epoch {epoch:03d}] loss={train_loss:.4f} | "
                  f"val DSC(mean)={mean_dsc:.4f}")

            if mean_dsc > best_dsc:
                best_dsc = mean_dsc
                ckpt_path = os.path.join(checkpoint_dir, "best_model.pth")
                # Unwrap DataParallel before saving — inference loads plain MambaBTS
                state = (model.module.state_dict()
                         if isinstance(model, nn.DataParallel)
                         else model.state_dict())
                torch.save({
                    "epoch": epoch,
                    "model_state": state,
                    "optimizer_state": optimizer.state_dict(),
                    "best_dsc": best_dsc,
                }, ckpt_path)
                print(f"  ↳ New best DSC={best_dsc:.4f}. Saved → {ckpt_path}")
        else:
            print(f"[Epoch {epoch:03d}] loss={train_loss:.4f}")

    print(f"\nTraining complete. Best validation DSC: {best_dsc:.4f}")
    return model


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 7 — INFERENCE
# ─────────────────────────────────────────────────────────────────────────────


def run_inference(
    checkpoint_path: str,
    modality_paths: dict,
    output_path: str = "prediction.nii.gz",
    patch_size: Tuple[int, int, int] = (96, 96, 96),
    num_classes: int = 4,
    base_features: int = 32,
    d_state: int = 16,
    device_str: str = "auto",
) -> np.ndarray:
    if device_str == "auto":
        device = (torch.device("cuda") if torch.cuda.is_available()
                  else torch.device("cpu"))
    else:
        device = torch.device(device_str)

    model = MambaBTS(in_channels=4, num_classes=num_classes,
                     base_features=base_features, d_state=d_state)
    ckpt = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(ckpt["model_state"])
    model.to(device).eval()

    preprocessor = BraTSPreprocessor()
    volume, _ = preprocessor.preprocess_subject(modality_paths)
    volume_t = torch.from_numpy(volume).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(volume_t)
    pred = logits.argmax(dim=1).squeeze(0).cpu().numpy().astype(np.uint8)

    ref_img = nib.load(list(modality_paths.values())[0])
    pred_nii = nib.Nifti1Image(pred, affine=ref_img.affine,
                                header=ref_img.header)
    nib.save(pred_nii, output_path)
    print(f"Prediction saved to: {output_path}")
    return pred


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 8 — SMOKE TEST
# ─────────────────────────────────────────────────────────────────────────────


def smoke_test(device_str: str = "cpu"):
    print("=" * 60)
    print("MambaBTS Smoke Test")
    print("=" * 60)

    device = torch.device(device_str)
    B, C, H, W, D = 1, 4, 64, 64, 64
    num_classes = 4

    print("\n[1/4] Instantiating MambaBTS model...")
    model = MambaBTS(in_channels=C, num_classes=num_classes,
                     base_features=16, d_state=8).to(device)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Trainable parameters: {n_params:,}")

    print("\n[2/4] Forward pass on random input...")
    x = torch.randn(B, C, H, W, D, device=device)
    with torch.no_grad():
        logits = model(x)
    assert logits.shape == (B, num_classes, H, W, D), \
        f"Output shape mismatch: {logits.shape}"
    print(f"  Input  shape: {tuple(x.shape)}")
    print(f"  Output shape: {tuple(logits.shape)}  ✓")

    print("\n[3/4] Loss computation (Dice + CE)...")
    criterion = CombinedLoss(num_classes=num_classes)
    target = torch.randint(0, num_classes, (B, H, W, D), device=device)
    loss = criterion(logits, target)
    print(f"  Combined loss: {loss.item():.4f}  ✓")

    print("\n[4/4] Evaluation metrics (DSC + HD95)...")
    pred_np = logits.argmax(dim=1).squeeze(0).cpu().numpy()
    tgt_np = target.squeeze(0).cpu().numpy()
    dsc = dice_similarity_coefficient(pred_np, tgt_np, num_classes)
    hd95 = hausdorff_distance_95(pred_np, tgt_np, num_classes)
    print("  DSC results:")
    for k, v in dsc.items():
        print(f"    {k:12s}: {v:.4f}")
    print("  HD95 results (mm):")
    for k, v in hd95.items():
        val_str = f"{v:.2f}" if not np.isinf(v) else "inf"
        print(f"    {k:12s}: {val_str}")

    print("\n" + "=" * 60)
    print("Smoke test PASSED ✓")
    print("=" * 60)


In [ ]:
import sys
import torch

sys.path.insert(0, "/kaggle/working")
from mamba_bts_impl import smoke_test

device_str = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__} | device: {device_str}")
if device_str == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

smoke_test(device_str=device_str)


## 4. Training

In [ ]:
import os
import torch
from mamba_bts_impl import train

# Reduce CUDA allocator fragmentation (recommended for large 3-D volumes)
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

# ── GPU info ──────────────────────────────────────────────────────────────────
n_gpus = torch.cuda.device_count()
for i in range(n_gpus):
    props = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {props.name}  {props.total_memory // 1024**2} MB")
print(f"Total GPUs available: {n_gpus}")

torch.cuda.empty_cache()

# ── Hyperparameters ───────────────────────────────────────────────────────────
# T4 x2: 2 × 16 GB with DataParallel.  patch_size=(96,96,96) is the safe
# upper bound — the selective SSM scan uses O(L × D × N) memory per step
# rather than the previous O(L × D × N) pre-allocated slab, but 96^3 is
# still the practical ceiling for a T4 (128^3 OOMs even with the fix).
# batch_size=2 → one volume per GPU via DataParallel.

trained_model = train(
    data_root      = DATA_DIR,
    num_epochs     = 100,
    batch_size     = 2,       # 1 sample per T4 via DataParallel
    lr             = 1e-4,
    patch_size     = (96, 96, 96),
    num_classes    = 4,
    base_features  = 32,
    d_state        = 16,
    checkpoint_dir = CKPT_DIR,
    device_str     = "auto",
)


## 5. Inference on held-out subjects (BraTS2021_00495 & BraTS2021_00621)

In [ ]:
from mamba_bts_impl import run_inference, dice_similarity_coefficient, hausdorff_distance_95
import nibabel as nib
import numpy as np

CHECKPOINT = os.path.join(CKPT_DIR, "best_model.pth")

for subj_name in ["BraTS2021_00495", "BraTS2021_00621"]:
    subj_dir = os.path.join(TEST_DIR, subj_name)
    if not os.path.isdir(subj_dir):
        print(f"Subject directory not found: {subj_dir}")
        continue

    modality_paths = {
        mod: os.path.join(subj_dir, f"{subj_name}_{mod}.nii.gz")
        for mod in ["t1", "t1ce", "t2", "flair"]
    }
    output_path = os.path.join(WORK_DIR, f"{subj_name}_pred.nii.gz")

    print(f"\n── Inference: {subj_name} ──")
    pred = run_inference(
        checkpoint_path = CHECKPOINT,
        modality_paths  = modality_paths,
        output_path     = output_path,
        patch_size      = (96, 96, 96),
        num_classes     = 4,
        base_features   = 32,
        d_state         = 16,
        device_str      = "auto",
    )

    # Evaluate against ground-truth seg if available
    seg_path = os.path.join(subj_dir, f"{subj_name}_seg.nii.gz")
    if os.path.exists(seg_path):
        gt = nib.load(seg_path).get_fdata(dtype=np.float32).astype(np.uint8)
        # Crop/pad pred to gt shape for metric computation
        if pred.shape != gt.shape:
            import numpy as np
            # Pad pred back to original space (simple centre-pad)
            padded = np.zeros(gt.shape, dtype=np.uint8)
            slices = []
            for ps, gs in zip(pred.shape, gt.shape):
                if ps <= gs:
                    pad = (gs - ps) // 2
                    slices.append((slice(pad, pad + ps), slice(0, ps)))
                else:
                    start = (ps - gs) // 2
                    slices.append((slice(0, gs), slice(start, start + gs)))
            padded[slices[0][0], slices[1][0], slices[2][0]] = \
                pred[slices[0][1], slices[1][1], slices[2][1]]
            pred_eval = padded
        else:
            pred_eval = pred

        dsc  = dice_similarity_coefficient(pred_eval, gt, num_classes=4)
        hd95 = hausdorff_distance_95(pred_eval, gt, num_classes=4)

        print(f"  DSC  — WT={dsc['WT']:.4f}  TC={dsc['TC']:.4f}  ET={dsc['ET']:.4f}  mean={dsc['mean_dsc']:.4f}")
        hd_wt = hd95['WT']; hd_tc = hd95['TC']; hd_et = hd95['ET']
        print(f"  HD95 — WT={hd_wt:.2f}  TC={hd_tc:.2f}  ET={hd_et:.2f} mm")
    else:
        print("  (No ground-truth segmentation found for metric computation.)")

print("\nAll inference complete. Predictions saved to /kaggle/working/")

## 6. (Optional) Quick visualisation — axial slice overlay

In [ ]:
import matplotlib
matplotlib.use("Agg")  # headless
import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np

LABEL_COLORS = {
    0: [0,   0,   0,   0],    # background — transparent
    1: [255, 0,   0,   180],  # NCR/NET — red
    2: [0,   255, 0,   180],  # ED  — green
    3: [0,   0,   255, 180],  # ET  — blue
}

def overlay_seg(flair_path, pred_path, out_png, slice_idx=None):
    flair = nib.load(flair_path).get_fdata(dtype=np.float32)
    pred  = nib.load(pred_path).get_fdata(dtype=np.float32).astype(np.uint8)

    if slice_idx is None:
        # pick the axial slice with most tumour voxels
        tumour_per_slice = [(pred[:, :, z] > 0).sum() for z in range(pred.shape[2])]
        slice_idx = int(np.argmax(tumour_per_slice))

    flair_sl = flair[:, :, slice_idx]
    pred_sl  = pred[:, :, slice_idx]

    # Normalise FLAIR for display
    flair_norm = (flair_sl - flair_sl.min()) / (flair_sl.max() - flair_sl.min() + 1e-8)

    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(flair_norm.T, cmap="gray", origin="lower")
    axes[0].set_title(f"FLAIR (z={slice_idx})")
    axes[0].axis("off")

    rgb = np.zeros((*flair_norm.shape, 4), dtype=np.uint8)
    rgb[..., :3] = (flair_norm[..., None] * 255).astype(np.uint8)
    rgb[..., 3]  = 255
    for label_id, color in LABEL_COLORS.items():
        if label_id == 0:
            continue
        mask = pred_sl == label_id
        rgb[mask] = color

    axes[1].imshow(rgb.transpose(1, 0, 2), origin="lower")
    axes[1].set_title("Prediction overlay")
    axes[1].axis("off")

    plt.tight_layout()
    plt.savefig(out_png, dpi=120)
    plt.close()
    print(f"Saved: {out_png}")

for subj_name in ["BraTS2021_00495", "BraTS2021_00621"]:
    subj_dir  = os.path.join(TEST_DIR, subj_name)
    flair_p   = os.path.join(subj_dir, f"{subj_name}_flair.nii.gz")
    pred_p    = os.path.join(WORK_DIR,  f"{subj_name}_pred.nii.gz")
    out_png   = os.path.join(WORK_DIR,  f"{subj_name}_overlay.png")
    if os.path.exists(flair_p) and os.path.exists(pred_p):
        overlay_seg(flair_p, pred_p, out_png)